# DAY MAY HOC NOI TIENG VIET (Text-To-Speech)

Notebook nay huan luyen **giong noi rieng** cho tro ly ao cua ban.

| Muc do | Thoi gian | Can GPU? | Ket qua |
|---|---|---|---|
| A. Kho giong san (gTTS) | 2 phut | Khong | Giong Google, dung duoc ngay |
| B. Finetune tren VietTTS | 3-8 gio | Co | Giong nu mien Bac tu nhien |
| C. Finetune giong CUA BAN | 2-6 gio | Co | Giong giong het ban |

**Truoc khi chay:** Runtime > Change runtime type > **T4 GPU** (cho muc B va C).

Giay phep dataset VietTTS (NTT123): chi dung cho muc dich **thu nghiem va giao duc**.


---
## 0. Kiem tra GPU


In [ ]:
!nvidia-smi
import torch
print('GPU san sang:', torch.cuda.is_available())


---
## A. KHO GIONG SAN (khong can GPU, lam truoc cho co cai dung ngay)

Tao san file mp3 cho nhung cau tro ly hay noi. Tai ve, giai nen vao thu muc
`voice_cache/` canh `tts.py` la tro ly phan hoi tuc thi, khong can mang.


In [ ]:
!pip install -q gTTS

import csv, os
from gtts import gTTS

PHRASES = [
    'Xin chao, toi da san sang nhan lenh',
    'Vang, toi lam ngay',
    'Da xong',
    'Dang mo trinh duyet',
    'Dang mo ung dung',
    'Dang mo file',
    'Dang tim kiem giup ban',
    'Dang phat nhac',
    'Toi da dat nhac nho cho ban',
    'Den gio roi ban oi',
    'Xin loi, toi chua hieu y ban',
    'Ban co chac muon tat may khong',
    'Da huy lenh',
    'Tam biet ban nhe',
]
# LUU Y: hay thay bang cau CO DAU tieng Viet de giong doc dung.

os.makedirs('voice_cache', exist_ok=True)
rows = []
for i, p in enumerate(PHRASES, 1):
    name = f'phrase_{i:03d}.mp3'
    gTTS(text=p, lang='vi').save(os.path.join('voice_cache', name))
    rows.append([p, name])

with open('voice_cache/index.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f); w.writerow(['text','file']); w.writerows(rows)

!zip -qr voice_cache.zip voice_cache
from google.colab import files
files.download('voice_cache.zip')


---
## B. FINETUNE GIONG THAT TREN DATASET VietTTS

Dataset: https://github.com/NTT123/Vietnamese-Text-To-Speech-Dataset/releases/tag/v1

Khoang 11-12 gio thu am mot giong nu mien Bac, dinh dang wav + transcript.


In [ ]:
# B1. Tai dataset (vai GB, mat vai phut)
!wget -q --show-progress https://github.com/NTT123/Vietnamese-Text-To-Speech-Dataset/releases/download/v1/infore.zip
!unzip -q infore.zip -d /content/tts_data
!ls /content/tts_data | head -20


In [ ]:
# B2. Xem cau truc du lieu
import os
for dirpath, dirnames, filenames in os.walk('/content/tts_data'):
    wavs = [f for f in filenames if f.endswith('.wav')]
    if wavs or filenames:
        print(dirpath, '|', len(wavs), 'wav |', filenames[:3])


In [ ]:
# B3. Chuyen ve dinh dang LJSpeech: metadata.csv dang  id|cau|cau
import glob, os

SRC = '/content/tts_data'
OUT = '/content/ljspeech'
os.makedirs(f'{OUT}/wavs', exist_ok=True)

# Tim file transcript (tuy ban release co the la .txt rieng tung file
# hoac mot file tong). Doan duoi xu ly ca hai truong hop.
pairs = []
for wav in glob.glob(f'{SRC}/**/*.wav', recursive=True):
    txt = os.path.splitext(wav)[0] + '.txt'
    if os.path.exists(txt):
        with open(txt, encoding='utf-8') as f:
            pairs.append((os.path.basename(wav)[:-4], f.read().strip(), wav))

print('So cap (audio, text):', len(pairs))

import shutil
with open(f'{OUT}/metadata.csv', 'w', encoding='utf-8') as f:
    for fid, text, wav in pairs:
        shutil.copy(wav, f'{OUT}/wavs/{fid}.wav')
        f.write(f'{fid}|{text}|{text}\n')
print('Da tao', f'{OUT}/metadata.csv')


### B4. Chon framework de train

| Framework | Do kho | Ghi chu |
|---|---|---|
| **Coqui TTS** | De nhat | `pip install TTS`, co recipe VITS san |
| **vietTTS (NTT123)** | Trung binh | Toi uu san cho chinh dataset nay |
| Tacotron2 + HiFi-GAN | Kho hon | Chat luong cao, chinh config nhieu |

Nguoi moi nen bat dau bang Coqui TTS.


In [ ]:
# B5a. Cach 1: Coqui TTS - finetune tu checkpoint da co
!pip install -q TTS

# Xem cac model co san
from TTS.api import TTS
print([m for m in TTS().list_models() if 'vi' in m or 'multilingual' in m][:10])

# Vi du lenh train VITS tren dataset LJSpeech vua tao:
# !python -m TTS.bin.train_tts \
#     --config_path config.json \
#     --restore_path /path/to/checkpoint.pth
# (tao config.json theo recipe VITS cua Coqui, tro dataset toi /content/ljspeech)


In [ ]:
# B5b. Cach 2: dung repo goc vietTTS (toi uu cho dataset nay)
!git clone -q https://github.com/NTT123/vietTTS.git
%cd vietTTS
!pip install -q -e .
!ls
# Doc README trong repo de biet lenh train chinh xac cho phien ban moi nhat


---
## C. FINETUNE GIONG CUA CHINH BAN

Chi can **30-60 phut thu am** la du de finetune ra giong giong ban.

1. Tren may, chay: `python train_tts.py record --count 100`
2. Nen thu muc `my_voice` thanh `my_voice.zip`
3. Upload len day va finetune tu checkpoint o muc B

Meo thu am: phong yen tinh, mic cach mieng 15-20cm, doc deu giong,
khong doc qua nhanh, moi cau nen dai 3-10 giay.


In [ ]:
from google.colab import files
up = files.upload()          # chon my_voice.zip
!unzip -qo my_voice.zip -d /content/
!head -5 /content/my_voice/metadata.csv
!ls /content/my_voice/wavs | wc -l


In [ ]:
# Tien xu ly: chuan hoa ve 22050Hz mono - bat buoc cho hau het framework TTS
!pip install -q librosa soundfile
import glob, librosa, soundfile as sf

for path in glob.glob('/content/my_voice/wavs/*.wav'):
    y, sr = librosa.load(path, sr=22050, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)   # cat khoang lang dau/cuoi
    sf.write(path, y, 22050)
print('Da chuan hoa xong.')


---
## D. SAU KHI TRAIN XONG: GAN GIONG MOI VAO TRO LY

1. Tai model ve may, dat canh `tts.py`.
2. Mo `tts.py`, them ham moi:

```python
_custom_tts = None

def _speak_custom(text: str) -> bool:
    global _custom_tts
    try:
        from TTS.api import TTS
        if _custom_tts is None:
            _custom_tts = TTS(model_path='model.pth', config_path='config.json')
        _custom_tts.tts_to_file(text=text, file_path='out.wav')
        return _play_audio('out.wav')
    except Exception:
        return False
```

3. Trong ham `speak()`, dat `_speak_custom` len **dau** danh sach `order`.
4. Kiem tra: `python train_tts.py check`


---
## Loi khuyen thuc te

- Train TTS **nang hon nhieu** so vc train intent. Hay lam xong phan hieu y
  va thuc thi cho chay on dinh truoc, dung tam `pyttsx3` hoac kho giong san.
- Neu chi can giong hay ma khong can giong rieng: dung **muc A** la du,
  chat luong gTTS tieng Viet da rat tot.
- Colab free hay ngat sau ~4 gio. Nho luu checkpoint vao Google Drive:
  `from google.colab import drive; drive.mount('/content/drive')`
